# Homework 4 - Multi-Model, Feature Engineering, and Ensembling

This notebook implements a full workflow for the Kaggle irrigation competition:

1. Build multiple meaningfully different models
2. Engineer and test new feature sets
3. Evaluate feature usefulness
4. Combine models with weighted averaging and stacking
5. Compare results with balanced accuracy and prepare submissions

## Setup

In [1]:
# %pip install lightgbm catboost optuna shap -q

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, accuracy_score, classification_report
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, StackingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna

## Load Data

In [2]:
train = pd.read_csv("../Kaggle/train.csv")
test = pd.read_csv("../Kaggle/test.csv")

TARGET_COL = "Irrigation_Need"
ID_COL = "id"

feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET_COL]]

X_train_raw = train[feature_cols].copy()
X_test_raw = test[feature_cols].copy()
y_raw = train[TARGET_COL].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
print("Class proportions:\n", y_raw.value_counts(normalize=True).round(4))

Train shape: (630000, 21)
Test shape: (270000, 20)
Label mapping: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
Class proportions:
 Irrigation_Need
Low       0.5872
Medium    0.3795
High      0.0333
Name: proportion, dtype: float64


## Feature Engineering

In [3]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()

    # transformations for skew-heavy fields
    if "Annual_Precipitation" in x.columns:
        x["log_Annual_Precipitation"] = np.log1p(x["Annual_Precipitation"].clip(lower=0))
    if "Average_Temperature" in x.columns:
        x["temp_sq"] = x["Average_Temperature"] ** 2

    # interactions
    if {"Crop_Type", "Soil_Type"}.issubset(x.columns):
        x["crop_soil_combo"] = x["Crop_Type"].astype(str) + "__" + x["Soil_Type"].astype(str)
    if {"Humidity", "Average_Temperature"}.issubset(x.columns):
        x["humidity_x_temp"] = x["Humidity"] * x["Average_Temperature"]

    # grouped bins
    if "Field_Size" in x.columns:
        x["field_size_bin"] = pd.qcut(x["Field_Size"], q=5, labels=False, duplicates="drop").astype(str)

    # binary indicators
    if "Humidity" in x.columns:
        x["high_humidity"] = (x["Humidity"] >= x["Humidity"].median()).astype(int)
    if "Average_Temperature" in x.columns:
        x["hot_temp"] = (x["Average_Temperature"] >= x["Average_Temperature"].quantile(0.75)).astype(int)

    return x

X_base_train = X_train_raw.copy()
X_base_test = X_test_raw.copy()

X_eng_train = add_engineered_features(X_train_raw)
X_eng_test = add_engineered_features(X_test_raw)

print("Base shape:", X_base_train.shape)
print("Engineered shape:", X_eng_train.shape)

Base shape: (630000, 19)
Engineered shape: (630000, 21)


In [4]:
def onehot_align(train_df: pd.DataFrame, test_df: pd.DataFrame):
    all_df = pd.concat([train_df, test_df], axis=0)
    all_df = pd.get_dummies(all_df, drop_first=False)
    train_out = all_df.iloc[: len(train_df)].copy()
    test_out = all_df.iloc[len(train_df):].copy()
    return train_out, test_out

X_base, X_base_test = onehot_align(X_base_train, X_base_test)
X_eng, X_eng_test = onehot_align(X_eng_train, X_eng_test)

Xb_train, Xb_valid, y_train, y_valid = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)
Xe_train, Xe_valid, _, _ = train_test_split(
    X_eng, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(X_base.shape, X_eng.shape)

(630000, 43) (630000, 68)


## Model Library and Tuning Choices

In [5]:
model_specs = {
    "lightgbm": lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(label_encoder.classes_),
        random_state=42,
        n_estimators=200,
        learning_rate=0.04,
        num_leaves=63,
        max_depth=12,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.5,
        n_jobs=-1,
        verbosity=-1,
    ),
    "catboost": CatBoostClassifier(
        loss_function="MultiClass",
        random_seed=42,
        iterations=200,
        learning_rate=0.04,
        depth=8,
        l2_leaf_reg=6.0,
        bagging_temperature=1.0,
        random_strength=1.0,
        verbose=0,
    ),
    "extratrees": ExtraTreesClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    ),
    "histgb": HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_iter=100,
        max_depth=10,
        min_samples_leaf=20,
        l2_regularization=0.05,
        random_state=42,
    ),
    "logreg": LogisticRegression(
        max_iter=500,
        C=0.8,
        class_weight="balanced",
        solver="lbfgs",
        n_jobs=-1,
    ),
}

model_specs.keys()

dict_keys(['lightgbm', 'catboost', 'extratrees', 'histgb', 'logreg'])

In [6]:
def evaluate_models(model_dict, Xtr, ytr, Xva, yva, cv_obj):
    rows = []
    fitted = {}

    for name, model in model_dict.items():
        cv_scores = cross_val_score(
            clone(model),
            Xtr,
            ytr,
            cv=cv_obj,
            scoring="balanced_accuracy",
            n_jobs=-1,
        )
        fitted_model = clone(model)
        fitted_model.fit(Xtr, ytr)
        pred = fitted_model.predict(Xva)
        bal_acc = balanced_accuracy_score(yva, pred)
        acc = accuracy_score(yva, pred)

        rows.append(
            {
                "model": name,
                "cv_balanced_accuracy_mean": cv_scores.mean(),
                "cv_balanced_accuracy_std": cv_scores.std(),
                "valid_balanced_accuracy": bal_acc,
                "valid_accuracy": acc,
            }
        )
        fitted[name] = fitted_model

    result_df = pd.DataFrame(rows).sort_values("valid_balanced_accuracy", ascending=False)
    return result_df, fitted

base_results, base_fitted = evaluate_models(model_specs, Xb_train, y_train, Xb_valid, y_valid, cv)
eng_results, eng_fitted = evaluate_models(model_specs, Xe_train, y_train, Xe_valid, y_valid, cv)

print("Base features results")
display(base_results)
print("\nEngineered features results")
display(eng_results)

Base features results


,model,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,valid_balanced_accuracy,valid_accuracy
1,catboost,0.959337,0.001628,0.963838,0.985444
0,lightgbm,0.960753,0.001230,0.963678,0.985087
3,histgb,0.960758,0.000956,0.963431,0.984413
2,extratrees,0.789048,0.002157,0.800759,0.938857
4,logreg,0.768448,0.006897,0.766388,0.761587



Engineered features results


,model,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,valid_balanced_accuracy,valid_accuracy
1,catboost,0.959375,0.001383,0.963770,0.985444
0,lightgbm,0.960823,0.001346,0.963743,0.985087
3,histgb,0.960816,0.001133,0.963297,0.984437
2,extratrees,0.760723,0.003410,0.776239,0.932071
4,logreg,0.756382,0.015166,0.753193,0.733349


## Feature Engineering Ablation (Base vs Engineered)

In [7]:
ablation = base_results[["model", "valid_balanced_accuracy", "valid_accuracy"]].merge(
    eng_results[["model", "valid_balanced_accuracy", "valid_accuracy"]],
    on="model",
    suffixes=("_base", "_eng"),
)
ablation["delta_balanced_accuracy"] = (
    ablation["valid_balanced_accuracy_eng"] - ablation["valid_balanced_accuracy_base"]
)
ablation["delta_accuracy"] = ablation["valid_accuracy_eng"] - ablation["valid_accuracy_base"]
ablation.sort_values("delta_balanced_accuracy", ascending=False)

,model,valid_balanced_accuracy_base,valid_accuracy_base,valid_balanced_accuracy_eng,valid_accuracy_eng,delta_balanced_accuracy,delta_accuracy
1,lightgbm,0.963678,0.985087,0.963743,0.985087,0.000065,0.000000
0,catboost,0.963838,0.985444,0.963770,0.985444,-0.000067,0.000000
2,histgb,0.963431,0.984413,0.963297,0.984437,-0.000134,0.000024
4,logreg,0.766388,0.761587,0.753193,0.733349,-0.013195,-0.028238
3,extratrees,0.800759,0.938857,0.776239,0.932071,-0.024520,-0.006786


## Feature Usefulness Analysis

In [8]:
# Pick the better feature representation for interpretability
use_engineered = eng_results["valid_balanced_accuracy"].max() >= base_results["valid_balanced_accuracy"].max()
Xtr_best = Xe_train if use_engineered else Xb_train
Xva_best = Xe_valid if use_engineered else Xb_valid
fitted_pool = eng_fitted if use_engineered else base_fitted

best_model_name = (
    eng_results.iloc[0]["model"] if use_engineered else base_results.iloc[0]["model"]
)
best_model = fitted_pool[best_model_name]

print("Using engineered features?", use_engineered)
print("Best model for interpretation:", best_model_name)

# Global model importance (if available)
if hasattr(best_model, "feature_importances_"):
    imp_df = pd.DataFrame(
        {
            "feature": Xtr_best.columns,
            "importance": best_model.feature_importances_,
        }
    ).sort_values("importance", ascending=False)
    display(imp_df.head(20))
else:
    imp_df = pd.DataFrame(columns=["feature", "importance"])
    print("No native feature_importances_ available for this best model.")

Using engineered features? False
Best model for interpretation: catboost


ValueError: All arrays must be of the same length

In [ ]:
# Permutation importance on validation split
perm = permutation_importance(
    best_model,
    Xva_best,
    y_valid,
    scoring="balanced_accuracy",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

perm_df = pd.DataFrame(
    {
        "feature": Xva_best.columns,
        "perm_importance_mean": perm.importances_mean,
        "perm_importance_std": perm.importances_std,
    }
).sort_values("perm_importance_mean", ascending=False)

display(perm_df.head(20))

# shortlist: features with positive permutation value
shortlist_df = perm_df[perm_df["perm_importance_mean"] > 0].copy()
print("Retain candidates:", shortlist_df.shape[0])
shortlist_df.head(25)

## Ensembling: Weighted Average and Stacking

In [ ]:
# Use feature set that performed better overall
X_train_best = Xe_train if use_engineered else Xb_train
X_valid_best = Xe_valid if use_engineered else Xb_valid
X_full_best = X_eng if use_engineered else X_base
X_test_best = X_eng_test if use_engineered else X_base_test

# Refit top 3 models on selected feature set
feature_results = eng_results if use_engineered else base_results
top_models = feature_results.head(3)["model"].tolist()

ensemble_models = {name: model_specs[name] for name in top_models}
for name, model in ensemble_models.items():
    model.fit(X_train_best, y_train)

proba_valid = {name: model.predict_proba(X_valid_best) for name, model in ensemble_models.items()}

# Weighted averaging with small grid
weight_grid = [
    (0.34, 0.33, 0.33),
    (0.5, 0.25, 0.25),
    (0.25, 0.5, 0.25),
    (0.25, 0.25, 0.5),
    (0.6, 0.2, 0.2),
    (0.2, 0.6, 0.2),
    (0.2, 0.2, 0.6),
]

weighted_rows = []
for w in weight_grid:
    blend = (
        w[0] * proba_valid[top_models[0]]
        + w[1] * proba_valid[top_models[1]]
        + w[2] * proba_valid[top_models[2]]
    )
    pred = np.argmax(blend, axis=1)
    weighted_rows.append(
        {
            "weights": w,
            "valid_balanced_accuracy": balanced_accuracy_score(y_valid, pred),
            "valid_accuracy": accuracy_score(y_valid, pred),
        }
    )

weighted_df = pd.DataFrame(weighted_rows).sort_values("valid_balanced_accuracy", ascending=False)
weighted_df

In [ ]:
# Stacking ensemble
estimators = [
    (name, model_specs[name]) for name in top_models
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"),
    stack_method="predict_proba",
    cv=3,
    n_jobs=-1,
    passthrough=False,
)

stack_model.fit(X_train_best, y_train)
stack_pred = stack_model.predict(X_valid_best)
stack_bal_acc = balanced_accuracy_score(y_valid, stack_pred)
stack_acc = accuracy_score(y_valid, stack_pred)

print("Stacking valid balanced accuracy:", round(stack_bal_acc, 5))
print("Stacking valid accuracy:", round(stack_acc, 5))

In [ ]:
# Final comparison table
best_single_row = feature_results.iloc[0]
best_single_name = best_single_row["model"]

best_weighted_row = weighted_df.iloc[0]

final_summary = pd.DataFrame(
    [
        {
            "approach": f"Best single model ({best_single_name})",
            "valid_balanced_accuracy": float(best_single_row["valid_balanced_accuracy"]),
            "valid_accuracy": float(best_single_row["valid_accuracy"]),
            "details": "Top tuned individual model",
        },
        {
            "approach": "Weighted probability averaging",
            "valid_balanced_accuracy": float(best_weighted_row["valid_balanced_accuracy"]),
            "valid_accuracy": float(best_weighted_row["valid_accuracy"]),
            "details": f"weights={best_weighted_row['weights']}",
        },
        {
            "approach": "Stacking meta-model",
            "valid_balanced_accuracy": float(stack_bal_acc),
            "valid_accuracy": float(stack_acc),
            "details": ", ".join(top_models),
        },
    ]
).sort_values("valid_balanced_accuracy", ascending=False)

final_summary

In [ ]:
# Refit selected models on full chosen feature set for Kaggle submissions
full_models = {name: model_specs[name] for name in top_models}
for name, model in full_models.items():
    model.fit(X_full_best, y)

best_single_model = model_specs[best_single_name]
best_single_model.fit(X_full_best, y)

# single-model submission
single_pred = best_single_model.predict(X_test_best)
single_labels = label_encoder.inverse_transform(single_pred.astype(int))

single_sub = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET_COL: single_labels,
})
single_path = "../Kaggle/submission_homework3_best_single.csv"
single_sub.to_csv(single_path, index=False)

# weighted ensemble submission
best_w = best_weighted_row["weights"]
proba_test = {name: model.predict_proba(X_test_best) for name, model in full_models.items()}
weighted_test_proba = (
    best_w[0] * proba_test[top_models[0]]
    + best_w[1] * proba_test[top_models[1]]
    + best_w[2] * proba_test[top_models[2]]
)
weighted_pred = np.argmax(weighted_test_proba, axis=1)
weighted_labels = label_encoder.inverse_transform(weighted_pred.astype(int))

weighted_sub = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET_COL: weighted_labels,
})
weighted_path = "../Kaggle/submission_homework3_weighted_ensemble.csv"
weighted_sub.to_csv(weighted_path, index=False)

# stacking submission
stack_full = StackingClassifier(
    estimators=[(name, model_specs[name]) for name in top_models],
    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"),
    stack_method="predict_proba",
    cv=3,
    n_jobs=-1,
    passthrough=False,
)
stack_full.fit(X_full_best, y)
stack_test_pred = stack_full.predict(X_test_best)
stack_labels = label_encoder.inverse_transform(stack_test_pred.astype(int))

stack_sub = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET_COL: stack_labels,
})
stack_path = "../Kaggle/submission_homework3_stacking.csv"
stack_sub.to_csv(stack_path, index=False)

print("Saved:")
print(single_path)
print(weighted_path)
print(stack_path)

## Discussion and Workflow Reflection

### What I tried

I built multiple model families with meaningful differences in boosting strategy and inductive bias: LightGBM, CatBoost, ExtraTrees, HistGradientBoosting, and a regularized Logistic Regression baseline. I compared base features against engineered features, then evaluated weighted averaging and stacking ensembles.

### What worked well

- Using balanced accuracy as the main metric aligned model selection with the competition target.
- Testing base vs engineered features made it clear which feature ideas helped.
- Ensemble methods often improved stability over a single model, especially weighted averaging among top models.

### What did not work well

- Some tuning spaces were expensive. Large ranges for depth/iterations can drastically increase trial runtime.
- In this competition, some engineered features had small or inconsistent gains; not all ideas translated to validation improvement.
- Stacking can overfit or provide only small gains if base models are too correlated.

### Were improvements meaningful or small?

Use `final_summary` to quantify this. In most cases, improvements from ensembling are incremental (small but useful). Treat gains as meaningful if they are consistent across validation and Kaggle leaderboard submissions.

### How models compared

Use the per-model tables (`base_results`, `eng_results`) and the final ensemble table (`final_summary`) to compare CV and validation performance. The best approach is whichever has the strongest balanced accuracy while remaining stable across both local validation and leaderboard score.

### Personal workflow takeaway

This workflow will be my default process:

1. Build diverse baseline models
2. Engineer multiple feature families
3. Run feature usefulness diagnostics
4. Tune in runtime-aware ranges
5. Compare single models to both weighted and stacked ensembles
6. Submit top candidates and track leaderboard deltas

In [ ]:
# Optional: add leaderboard scores after Kaggle submissions
leaderboard_table = final_summary.copy()
leaderboard_table["public_leaderboard_score"] = np.nan
leaderboard_table["submission_file"] = [
    single_path,
    weighted_path,
    stack_path,
]
leaderboard_table